## Setup

In [2]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

In [3]:
chapter_name = "setup"

from pyspark.sql import SparkSession

# Default memory will not be sufficient for the examples below, so we up it to 4GB per 
# executor
executor_memory = "4g"
executor_cores = 4
num_executors = 2


spark = SparkSession.builder \
        .appName(chapter_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
        .getOrCreate()

Setting Spark log level to "WARN".


In [4]:
# Create data and load into Iceberg tables for the code below
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

run(scaling_factor=1) # data will be 1GB on your disk
run_ddl(data_path=Path("./data"), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
spark.sql("use prod.db") # this is the schema in which our data is

DataFrame[]

## Partitioning

In [7]:
# FULL TABLE SCAN = All data is read into Spark Memory and then filtered
from pyspark.sql import functions as F

spark.read.table("lineitem").where(F.col("l_receiptdate") == "1992-01-05").explain()

== Physical Plan ==
*(1) Filter (isnotnull(l_receiptdate#589) AND (l_receiptdate#589 = 1992-01-05))
+- *(1) ColumnarToRow
   +- BatchScan demo.prod.db.lineitem[l_orderkey#577, l_partkey#578, l_suppkey#579, l_linenumber#580, l_quantity#581, l_extendedprice#582, l_discount#583, l_tax#584, l_returnflag#585, l_linestatus#586, l_shipdate#587, l_commitdate#588, l_receiptdate#589, l_shipinstruct#590, l_shipmode#591, l_comment#592] demo.prod.db.lineitem (branch=null) [filters=l_receiptdate IS NOT NULL, l_receiptdate = 8039, groupedBy=] RuntimeFilters: []




In [9]:
# create a partitioned table
spark\
.read\
.table("lineitem")\
.writeTo("lineitem_rdp")\
.partitionedBy("l_receiptdate")\
.createOrReplace()

In [10]:
# PARTITION PRUNING = Only read the necessary partition
spark\
.read\
.table("lineitem_rdp")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.show()

+----------+---------+---------+------------+----------+---------------+----------+-----+------------+------------+----------+------------+-------------+-----------------+----------+--------------------+
|l_orderkey|l_partkey|l_suppkey|l_linenumber|l_quantity|l_extendedprice|l_discount|l_tax|l_returnflag|l_linestatus|l_shipdate|l_commitdate|l_receiptdate|   l_shipinstruct|l_shipmode|           l_comment|
+----------+---------+---------+------------+----------+---------------+----------+-----+------------+------------+----------+------------+-------------+-----------------+----------+--------------------+
|    156868|   125430|     7943|           4|      10.0|        14554.3|      0.03| 0.08|           A|           F|1992-01-04|  1992-03-22|   1992-01-05|             NONE|      MAIL| the close, unusu...|
|    359170|    82015|     7032|           3|      40.0|        39880.4|      0.08| 0.02|           R|           F|1992-01-03|  1992-02-18|   1992-01-05|DELIVER IN PERSON|      RAIL|lu

## Bucketing

In [11]:
# NO BUCKETING GROUP BY = Exchange (movement of data between nodes in the cluster) 
# EXPENSIVE !!!
spark.sql("""
SELECT l_receiptdate
, sum(l_quantity) as total_quantity
FROM lineitem
GROUP BY l_receiptdate
""").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[l_receiptdate#777], functions=[sum(l_quantity#769)])
   +- Exchange hashpartitioning(l_receiptdate#777, 200), ENSURE_REQUIREMENTS, [plan_id=513]
      +- HashAggregate(keys=[l_receiptdate#777], functions=[partial_sum(l_quantity#769)])
         +- BatchScan demo.prod.db.lineitem[l_quantity#769, l_receiptdate#777] demo.prod.db.lineitem (branch=null) [filters=, groupedBy=] RuntimeFilters: []




In [12]:
# Create a bucketed table, with the bucketing based on the group by column
from pyspark.sql import functions as F

spark.read.table("lineitem")\
.writeTo("lineitem_bb_rd") \
.partitionedBy(  
    F.partitioning.bucket(4, "l_receiptdate")
).createOrReplace()

In [13]:
# BUCKETING = NO Exchange 
# settings to ensure Spark + Iceberg can leverage the underlying data layout
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')

spark.sql("""
SELECT l_receiptdate
, sum(l_quantity) as total_quantity
FROM lineitem_bb_rd
GROUP BY l_receiptdate
""").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[l_receiptdate#869], functions=[sum(l_quantity#861)])
   +- HashAggregate(keys=[l_receiptdate#869], functions=[partial_sum(l_quantity#861)])
      +- BatchScan demo.prod.db.lineitem_bb_rd[l_quantity#861, l_receiptdate#869] demo.prod.db.lineitem_bb_rd (branch=null) [filters=, groupedBy=l_receiptdate_bucket] RuntimeFilters: []




## Sorting

In [14]:
# Can only be noticed if we run the query
# NO SORTING = Large data read into Spark Memory
spark.sql("""
select *
from lineitem
where l_extendedprice between 3000 and 10000
""")\
.write\
.csv(f"./{chapter_name}/lineitem", mode = 'overwrite')

In [15]:
# Create Sorted Table
spark.read.table("lineitem")\
.sort("l_extendedprice")\
.coalesce(3)\
.writeTo("lineitem_sb_ep")\
.createOrReplace()

In [16]:
# SORTING = Smaller data read into Spark Memory
spark.sql("""
select *
from lineitem_sb_ep
where l_extendedprice between 3000 and 10000
""")\
.write\
.csv(f"./{chapter_name}/lineitem_0rd", mode = 'overwrite')